# 19.22 助理的預測怎樣成為真正可播放的聲音？


短片裡發聲時的形狀若是圓，讓助理發出「440Hz、0.2秒 → 停0.1秒 → 880Hz、0.2秒」；若是方，兩個音高反過來。這是公開約定的有限提示音，不是真人語音合成，也沒有把「圓」念出來。它接起[19.21](https://birdhackor.github.io/tiny-perceptron-vlm/19.21.html)的影音答案、[17.16](https://birdhackor.github.io/tiny-perceptron-vlm/17.16.html)的 codec（聲音編解碼器），以及[16.16](https://birdhackor.github.io/tiny-perceptron-vlm/16.16.html)的因果輸出：答案要一路變成波形，才有可交給播放器的樣本。

先凍結 semantic（語義結果）到音型的字典。`circle`、`square`、`unknown`分別是圓、方、不可判定；最後一種明確轉交，不產生靜音冒充有解成功。正式流程取 A 影音 head 的真預測作 **canonical semantic result（後續共同採用的語義結果）**；共享對話可以記錄它，文字模型潤飾的句子不能覆寫它。若正解是方，A卻預測圓，就必須沿圓的條件生成，讓驗收看見上游錯誤。

圖中的第一框是正式要接的 A 介面。正式字典把圓轉成「220Hz發聲200ms、停100ms、440Hz發聲200ms」的音型程式，方則把第一音換成880Hz。Talker讀程式的公開條件並預測聲音碼，凍結的 decoder 再將碼還原成8kHz波形。gold（標註正解）另供對照與評分，沒有通路把 A 的錯答換成正解。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/omni-19.22-sound-chain.svg")))

下面把必要步驟完整攤開，先用兩個條件ID與另一份簡化音型字典。codec 採[12.18](https://birdhackor.github.io/tiny-perceptron-vlm/12.18.html)的固定查表簡化：一碼代表100ms，三列是靜音、440Hz、880Hz，所以小例中圓的目標為 `[1,1,0,2,2]`。hop為800樣本，沒有跨塊 state 或尾部，lookahead為0；這不是圖中的正式音型或正式 codec 的已選超參數。正式版要先在開發集選定 encoder、碼本、decoder與前處理並凍結hash，才產生 Talker targets，之後不再更新碼本或統計。

這個 Talker 用 embedding 把碼與條件查成向量；**GRU（Gated Recurrent Unit，門控循環單元）**每步把目前碼向量與舊 state 合成新 state，保存先前生成內容；Linear 再替下一碼打分。state有8個數，由條件向量初始化，每次生成重新開始。聲音碼是0–2，EOS=3表示結束，BOS=4表示開始，輸出頭不能產生BOS。

訓練使用[7.4](https://birdhackor.github.io/tiny-perceptron-vlm/7.4.html)的下一位置對齊，叫 **teacher forcing（教師強制）**：輸入BOS及真正的前碼，學下一碼與EOS。生成時只有BOS，之後每步接回自己的 `argmax` 預測，直到EOS或上限；`generate`根本不收 targets。程式只做一次CPU參數更新，不載入新版A，而以人工 `manual_pipeline_fixture` 紀錄故意填圓、gold填方，觀察兩條路是否分開。


In [ ]:
import math
import struct
import wave
import torch
from torch import nn
import torch.nn.functional as F

torch.set_num_threads(1)
torch.manual_seed(0)
sample_rate, hop = 8000, 800  # CPU 示意：一碼100ms，非正式超參數。
programs = {
    "circle": [(440, 0.2), (0, 0.1), (880, 0.2)],
    "square": [(880, 0.2), (0, 0.1), (440, 0.2)],
}
condition_ids = {"circle": 0, "square": 1}


def make_wave(program, phase=0.0):
    pieces = []
    for frequency, seconds in program:
        t = torch.arange(round(seconds * sample_rate)) / sample_rate
        piece = torch.zeros_like(t) if frequency == 0 else 0.2 * torch.sin(2 * math.pi * frequency * t + phase)
        pieces.append(piece)
    return torch.cat(pieces)


class BlockCodec(nn.Module):
    def __init__(self):
        super().__init__()
        # 和12.18一樣：原始區塊直接當特徵，查表直接當decoder。
        t = torch.arange(hop) / sample_rate
        self.table = nn.Parameter(
            torch.stack(
                [
                    torch.zeros(hop),
                    0.2 * torch.sin(2 * math.pi * 440 * t),
                    0.2 * torch.sin(2 * math.pi * 880 * t),
                ]
            )
        )

    def encode(self, wave):
        if wave.numel() % hop:
            raise ValueError("示意codec只接受完整區塊")
        blocks = wave.reshape(-1, hop)
        distance = (blocks[:, None] - self.table[None]).square().mean(-1)
        return distance.argmin(-1)

    def decode(self, codes):
        # 每碼獨立輸出800樣本，lookahead=0，沒有跨塊state或尾部。
        return self.table[torch.tensor(codes, dtype=torch.long)].flatten()


codec = BlockCodec().eval()
codec.requires_grad_(False)  # 必須在產生Talker targets之前。
frozen_table = codec.table.detach().clone()
K, EOS, BOS = 3, 3, 4
with torch.no_grad():
    targets = {}
    for semantic, program in programs.items():
        # Talker的示範音訊固定每個事件初相位為0。
        codes = codec.encode(make_wave(program, phase=0.0))
        targets[semantic] = torch.cat((codes, torch.tensor([EOS])))
print("凍結後targets（含EOS）", {k: v.tolist() for k, v in targets.items()})


class Talker(nn.Module):
    def __init__(self):
        super().__init__()
        self.token = nn.Embedding(K + 2, 8)  # 聲音碼、EOS、BOS。
        self.condition = nn.Embedding(2, 8)
        self.gru = nn.GRU(8, 8, batch_first=True)
        self.head = nn.Linear(8, K + 1)  # 只能輸出3個碼或EOS。

    def start(self, condition):
        return self.condition(condition).unsqueeze(0)

    def forward(self, token_ids, state):
        hidden, state = self.gru(self.token(token_ids), state)
        return self.head(hidden), state


talker = Talker()
optimizer = torch.optim.SGD(talker.parameters(), lr=0.01)  # 不含codec。
talker.train()
labels = torch.stack([targets["circle"], targets["square"]])
conditions = torch.tensor([0, 1])
train_inputs = torch.cat((torch.full((2, 1), BOS), labels[:, :-1]), dim=1)
logits, _ = talker(train_inputs, talker.start(conditions))
loss = F.cross_entropy(logits.reshape(-1, K + 1), labels.flatten())
optimizer.zero_grad()
loss.backward()
optimizer.step()  # 只做一次CPU機制更新，沒有訓練成品。
print("teacher forcing首列輸入／目標", train_inputs[0].tolist(), labels[0].tolist())
print("一次更新前的交叉熵", round(loss.item(), 6))
print("codec保持相同", torch.equal(frozen_table, codec.table))


@torch.no_grad()
def generate(condition, max_steps=6):
    talker.eval()
    state = talker.start(torch.tensor([condition]))
    token = torch.tensor([[BOS]])
    codes = []
    for _ in range(max_steps):  # 最多6次預測，包括可能的EOS。
        logits, state = talker(token, state)
        next_id = logits[0, -1].argmax().item()
        if next_id == EOS:
            return codes, "EOS"
        codes.append(next_id)
        token = torch.tensor([[next_id]])  # 只接回自己的預測。
    return codes, "max_steps"


def map_result(result):
    if not isinstance(result, dict) or not isinstance(result.get("semantic"), str):
        raise ValueError("semantic必須是字串欄位")
    semantic = result.get("semantic")
    if semantic == "unknown":
        return None  # 明確轉交，不產生靜音冒充有解成功。
    if semantic not in programs:
        raise ValueError("無法解析或不支援的semantic")
    return condition_ids[semantic]


def save_wav(path, waveform):
    values = waveform.tolist()
    if not values or not all(math.isfinite(v) and abs(v) <= 1 for v in values):
        raise ValueError("WAV需要非空、有限、振幅介於-1和1的波形")
    # PCM16每點用2 bytes＝16位元；把[-1,1]振幅映成[-32767,32767]的整數。
    samples = [round(v * 32767) for v in values]
    with wave.open(path, "wb") as output:
        output.setnchannels(1)
        output.setsampwidth(2)
        output.setframerate(sample_rate)
        # "<"是低位元組先寫的小端序；每個"h"是一個有符號16位元整數。
        # 重複h指定樣本數；*samples將清單逐項交給pack，產生可寫入WAV的bytes。
        output.writeframes(struct.pack("<" + "h" * len(samples), *samples))


def run_case(case):
    # gold只建立獨立對照；fixture分支只讀a_result。
    gold_codes, gold_stop = generate(condition_ids[case["gold"]])
    gold_run = {"codes": gold_codes, "stop": gold_stop, "waveform": codec.decode(gold_codes)}
    try:
        condition = map_result(case["a_result"])
    except ValueError:
        return {"gold": gold_run, "chain": {"status": "parse_error"}}
    if condition is None:
        return {"gold": gold_run, "chain": {"status": "transfer"}}
    codes, stop = generate(condition)
    return {
        "gold": gold_run,
        "chain": {"status": "generated", "codes": codes, "stop": stop, "waveform": codec.decode(codes)},
    }


# 人工管線fixture，沒有執行新版A；故意讓紀錄和gold不同。
a_result = {"semantic": "circle", "source": "manual_pipeline_fixture"}
panel = [{"gold": "square", "a_result": a_result}]
rows = [run_case(case) for case in panel]  # 不先篩掉A錯答／轉交／解析失敗。
print("人工A紀錄／gold", a_result["semantic"], panel[0]["gold"])
for branch in ["gold", "chain"]:
    result = rows[0][branch]
    if "waveform" in result:
        print(branch, result["codes"], result["stop"], "樣本數", result["waveform"].numel())
    else:
        print(branch, result["status"], "沒有合成waveform")
print("示意記帳分母 gold／整鏈", len(panel), len(rows))
if "waveform" in rows[0]["chain"] and rows[0]["chain"]["waveform"].numel():
    save_wav("demo-sound.wav", rows[0]["chain"]["waveform"])
    print("寫出 demo-sound.wav（示意生成，音型未成功）")

關鍵輸出如下；3是EOS，並不是第四個聲音碼：

```text
凍結後targets（含EOS） {'circle': [1, 1, 0, 2, 2, 3], 'square': [2, 2, 0, 1, 1, 3]}
teacher forcing首列輸入／目標 [4, 1, 1, 0, 2, 2] [1, 1, 0, 2, 2, 3]
一次更新前的交叉熵 1.371999
codec保持相同 True
人工A紀錄／gold circle square
gold [2] EOS 樣本數 800
chain [2] EOS 樣本數 800
示意記帳分母 gold／整鏈 1 1
```

兩條路都只生成100ms的880Hz音便結束，缺少另一事件、必要停頓與完整時值。這個只更新一次的小模型沒有完成音型；EOS正常、波形存在，不等於生成成功。末尾將它寫成目前資料夾的 `demo-sound.wav`，可用一般播放器開啟；這裡收齊結果才寫檔，沒有量測首段等待。若直接把 `[1,1,0,2,2]`送去解碼，只是在播放參考碼的重建，不能算 Talker 自由生成。`run_case`即使遇到轉交或解析錯誤仍回傳一筆紀錄，所以沒有因為 A 出錯便縮小整鏈分母。

正式驗收要分三種工作：codec 在參考事件區間比較原波形與重建，SNR（signal-to-noise ratio，訊雜比）看原波形能量相對於重建誤差能量的比例，10dB約對應誤差能量為原能量十分之一；gold條件測預定聲音家族的自主生成，隔開A的錯誤；真A整鏈則在每個有解final家族的預指定題，實跑 A→映射→自由生成→decoder→分析器，依**正解音型**判對錯，分母包含A錯答、拒答、解析錯、靜音與生成錯。另報A、映射、真條件下的聲音表現及合法率，才能看出是答錯、映錯，還是聲音沒生好。原本應答unknown的刪線索題另報，不能用靜音提高有解整鏈成績。

分析器要在開發階段用正例，以及錯頻率、倒序、錯節奏、漏／多事件、全靜音負例固定下來。生成檢查頻率、順序、起訖、時值及停頓，整個程式都符合才成功；預設頻率容忍5%、時間容忍30ms。重建SNR不重對齊、不調增益，各非靜音程式先計算再按家族等權平均，純靜音另報誤發聲。SNR≥10dB、gold條件成功≥85%、真A整鏈≥80%，以及8kHz、有限數值、格式／碼值合法、≤2秒且≤4事件的合法率100%，都是在實測前固定的驗收門檻，前兩項不能推導整鏈通過。

資料以完整「頻率、事件時值、停頓」組合作家族，phase（初相位）、振幅與雜訊變體留在同一split，codec與Talker訓練都不能見到final組合。Talker目標可固定公開的初相位0，讓同條件有穩定目標；重建仍須對原相位評分，不能偷偷重設。若音型已在Talker訓練看過，整鏈只能稱未見影音輸入接已見音型，新音型組合須另測。本例只有兩個條件，不提供留出能力證據；正式架構、資料及條件時程另列於本節後段。

練習只把人工 `a_result` 的 `semantic` 改成 `unknown`，從頭執行。先預測 chain 變成 `transfer`、沒有waveform，而gold路徑仍是 `[2]`、EOS，兩個記帳分母仍為1；再把它改成不支援的字串，確認記為 `parse_error`。在有解題上，轉交與解析失敗都留在整鏈分母，不能靠刪掉難題提高成功率。

**正式模型把哪些簡化換掉了？** 上面的查表與兩個條件ID是機制練習。正式codec從隨機權重學習；Talker讀的條件也不只是一個圓／方ID，而是使用者提交的完整有限音型程式。下表的MLP是多層感知機（multilayer perceptron），這裡指線性層與非線性串接的小網路；表中的維度箭頭列出每層輸入、輸出的格數。

| 零件 | 上面的CPU小例 | 正式訓練版 |
| --- | --- | --- |
| codec | 三個人工100 ms區塊 | 8 kHz、80 samples＝10 ms一碼；32維latent、256個可學碼本向量 |
| encoder／decoder | 區塊本身／查表波形 | `80→128→32`與`32→128→80`的MLP；decoder末端Tanh限制振幅 |
| 條件 | 圓／方兩個ID | 公開頻率、段內位置、事件起點、END；不讀參考波形或參考聲音碼 |
| Talker | 8維GRU | 96維GRU，BOS=256、EOS=257、PAD=258；最多200個聲音碼 |
| 正式語義音型 | 不使用小例字典驗收 | 圓＝220 Hz 200 ms、停100 ms、440 Hz 200 ms；方把第一音換成880 Hz |

碼本查詢仍是[17.16](https://birdhackor.github.io/tiny-perceptron-vlm/17.16.html)的最近向量與straight-through。loss是有效樣本的重建MSE、碼本誤差，加上0.25倍commitment誤差；全padding區塊不參與。codec訓練完成後先選dev checkpoint、凍結參數與hash，才產生Talker targets。Talker每步額外知道輸入音型要求的頻率與時間位置；END告訴它要求的片長，但仍須自己輸出EOS，不能按gold長度直接截成成功。因此這是**有限音型程式到聲音碼**，不能推廣成未受限制的文字到語音。

<details>
<summary>正式codec、公開時程與Talker完整計算</summary>

下面直接列出正式模型的前向與生成計算。只省略訓練時的未用碼重啟統計；那不是推論所需程式。`Event`描述一個發聲事件及其後停頓；公開時程可由輸入程式計算，不會把waveform或參考碼偷偷帶入。最後的形狀核對使用人工碼列和隨機新參數，不載入正式權重。


In [ ]:
from dataclasses import dataclass
import torch
from torch import nn
import torch.nn.functional as F


@dataclass(frozen=True)
class Event:
    frequency: int
    duration_ms: int
    rest_ms: int = 0


def validate_program(program):
    if not 1 <= len(program) <= 4 or any(type(x) is not Event for x in program):
        raise ValueError("聲音程式須為 1–4 個 Event")
    if any(
        x.frequency not in (220, 440, 880) or x.duration_ms not in (100, 200, 300) or x.rest_ms not in (0, 100, 200)
        for x in program
    ):
        raise ValueError("頻率／時值／停頓不在公開有限契約")
    if program[-1].rest_ms != 0 or sum(x.duration_ms + x.rest_ms for x in program) > 2000:
        raise ValueError("片尾無額外停頓且總長不超過 2 秒")
    if any(a.frequency == b.frequency and a.rest_ms == 0 for a, b in zip(program, program[1:])):
        raise ValueError("同頻零停頓的兩事件不可觀察，須合併後另定 family")
    return program


@dataclass(frozen=True)
class CodecConfig:
    hop: int = 80
    width: int = 32
    codes: int = 128
    hidden: int = 128
    beta: float = 0.25
    sample_rate: int = 8000

    def validate(self):
        if self.sample_rate != 8000 or self.hop not in (40, 80) or self.codes not in (32, 64, 128, 256):
            raise ValueError("codec profile 不在事先公開的候選集合")
        if self.width not in (16, 32, 64) or self.beta != 0.25:
            raise ValueError("latent width／commitment beta 不合法")
        return self


class SoundCodec(nn.Module):
    """每個 block 零 lookahead；最後 padding 不進 loss／碼本統計。"""

    def __init__(self, config=CodecConfig()):
        super().__init__()
        self.config = config.validate()
        self.encoder = nn.Sequential(
            nn.Linear(config.hop, config.hidden), nn.GELU(), nn.Linear(config.hidden, config.width)
        )
        self.codebook = nn.Embedding(config.codes, config.width)
        nn.init.normal_(self.codebook.weight, std=0.15)
        self.decoder = nn.Sequential(
            nn.Linear(config.width, config.hidden), nn.GELU(), nn.Linear(config.hidden, config.hop), nn.Tanh()
        )

    def quantize(self, encoded):
        weights = self.codebook.weight
        distances = encoded.square().sum(-1, keepdim=True) + weights.square().sum(-1) - 2 * encoded @ weights.T
        codes = distances.argmin(-1)
        return self.codebook(codes), codes

    def forward(self, blocks, valid=None):
        if blocks.shape[-1] != self.config.hop:
            raise ValueError("codec blocks 的 hop 不符")
        if valid is None:
            valid = torch.ones_like(blocks, dtype=torch.bool)
        if valid.shape != blocks.shape or valid.dtype != torch.bool:
            raise ValueError("sample padding mask 不符")
        encoded = self.encoder(torch.where(valid, blocks, 0))
        quantized, codes = self.quantize(encoded)
        straight_through = encoded + (quantized - encoded).detach()
        reconstructed = self.decoder(straight_through)
        nonpadding = valid.any(-1)
        if not nonpadding.any() or not valid.any():
            raise ValueError("全 padding 不應進 codec loss")
        reconstruction = (reconstructed[valid] - blocks[valid]).square().mean()
        codebook_loss = (quantized - encoded.detach()).square()[nonpadding].mean()
        commitment = (encoded - quantized.detach()).square()[nonpadding].mean()
        return {
            "waveform": reconstructed,
            "codes": torch.where(nonpadding, codes, -1),
            "loss": reconstruction + codebook_loss + self.config.beta * commitment,
            "reconstruction_loss": reconstruction,
            "codebook_loss": codebook_loss,
            "commitment_loss": commitment,
            "encoded": encoded[nonpadding],
        }

    @torch.no_grad()
    def encode_wave(self, waveform):
        length, hop = len(waveform), self.config.hop
        if waveform.ndim != 1 or not length or not torch.isfinite(waveform).all():
            raise ValueError("waveform 非法")
        padded = F.pad(waveform, (0, (-length) % hop))
        _, codes = self.quantize(self.encoder(padded.reshape(-1, hop)))
        return codes, length

    def decode_codes(self, codes):
        if codes.ndim != 1 or not len(codes) or codes.min() < 0 or codes.max() >= self.config.codes:
            raise ValueError("codec code 超限或為空")
        # 逐塊與整段走完全相同的矩陣形狀／次序，CPU 與 GPU 皆可逐樣本對照。
        return torch.cat([self.decoder(self.codebook(code)).reshape(-1) for code in codes])

    def chunks(self, codes):
        for code in codes:
            yield self.decoder(self.codebook(code)).reshape(-1)


def program_schedule(program, hop=80, max_blocks=None):
    """只由公開輸入程式推導的時程，沒有參考波形、codec 或參考 codes。

    每列是 requested frequency／segment 內 block 位置／事件起點。
    最後 END 列是輸入所要求的片長特徵；Talker 仍須自己預測 EOS。
    """
    validate_program(program)
    if hop not in (40, 80):
        raise ValueError("Talker 公開時程僅支援已定義的 hop40/80")
    max_blocks = 16000 // hop if max_blocks is None else max_blocks
    rows = []
    frequencies = {0: 0, 220: 1, 440: 2, 880: 3}
    for event in program:
        for position in range(event.duration_ms * 8 // hop):
            rows.append((frequencies[event.frequency], position, int(position == 0)))
        for position in range(event.rest_ms * 8 // hop):
            rows.append((0, position, int(position == 0)))
    if len(rows) > max_blocks:
        raise ValueError("聲音程式超過指定 blocks 上限")
    rows.append((4, 0, 1))
    return torch.tensor(rows, dtype=torch.long)


class ProgramTalker(nn.Module):
    def __init__(self, codes=128, width=96, hop=80):
        super().__init__()
        if hop not in (40, 80):
            raise ValueError("Talker hop 不合法")
        self.codes, self.width, self.hop = codes, width, hop
        self.max_blocks = 16000 // hop
        self.bos, self.eos, self.pad = codes, codes + 1, codes + 2
        self.code_embedding = nn.Embedding(codes + 3, width)
        self.frequency_embedding = nn.Embedding(5, width)
        self.position_embedding = nn.Embedding(2400 // hop, width)
        self.onset_embedding = nn.Embedding(2, width)
        self.recurrent = nn.GRU(width, width, batch_first=True)
        self.output = nn.Linear(width, codes + 2)

    def inputs(self, previous, schedule):
        return (
            self.code_embedding(previous)
            + self.frequency_embedding(schedule[..., 0])
            + self.position_embedding(schedule[..., 1])
            + self.onset_embedding(schedule[..., 2])
        )

    def forward(self, previous, schedule):
        hidden, _ = self.recurrent(self.inputs(previous, schedule))
        return self.output(hidden)

    @torch.no_grad()
    def generate(self, program, max_blocks=None):
        """僅 BOS 起始，之後只讀自己的預測；不收 reference codes／length。"""
        device = self.code_embedding.weight.device
        max_blocks = self.max_blocks if max_blocks is None else max_blocks
        if type(max_blocks) is not int or not 1 <= max_blocks <= self.max_blocks:
            raise ValueError("生成 blocks 上限非法")
        schedule = program_schedule(program, self.hop).to(device)
        previous = torch.tensor([[self.bos]], device=device)
        hidden, emitted = None, []
        for step in range(max_blocks + 1):
            # 超過公開意圖片長仍讀 END；不直接用它終止，不補／截成 gold 長度。
            row = schedule[min(step, len(schedule) - 1)].reshape(1, 1, 3)
            value, hidden = self.recurrent(self.inputs(previous, row), hidden)
            predicted = self.output(value[:, -1]).argmax(-1).item()
            if predicted == self.eos:
                return {"codes": torch.tensor(emitted, device=device, dtype=torch.long), "eos": True, "capped": False}
            if predicted == self.bos or len(emitted) == max_blocks:
                return {"codes": torch.tensor(emitted, device=device, dtype=torch.long), "eos": False, "capped": True}
            emitted.append(predicted)
            previous = torch.tensor([[predicted]], device=device)
        raise RuntimeError("unreachable")

    def config(self):
        return {
            "codes": self.codes,
            "width": self.width,
            "hop": self.hop,
            "sample_rate": 8000,
            "max_blocks": self.max_blocks,
            "conditioning": "public-input-derived frequency/elapsed/onset/END schedule; no reference codes or forced stopping length",
        }


torch.set_num_threads(1)
torch.manual_seed(20261029)
formal_codec = SoundCodec(CodecConfig(codes=256)).eval().requires_grad_(False)
formal_talker = ProgramTalker(codes=256, width=96, hop=80).eval()
formal_program = (Event(220, 200, 100), Event(440, 200))
formal_schedule = program_schedule(formal_program)
# 任意合法50個聲音碼作純形狀核對；不是此音型的gold或模型成績。
example_codes = torch.arange(50) % 256
example_target = torch.cat([example_codes, torch.tensor([formal_talker.eos])])
example_previous = torch.cat([torch.tensor([formal_talker.bos]), example_codes])
with torch.no_grad():
    formal_scores = formal_talker(example_previous[None], formal_schedule[None])
    formal_loss = F.cross_entropy(formal_scores.flatten(0, 1), example_target)
    whole = formal_codec.decode_codes(example_codes)
    chunked = torch.cat(list(formal_codec.chunks(example_codes)))
print("BOS／EOS／PAD", formal_talker.bos, formal_talker.eos, formal_talker.pad)
print("公開時程首／末", formal_schedule[0].tolist(), formal_schedule[-1].tolist())
print("teacher forcing形狀", tuple(formal_scores.shape), tuple(example_target.shape))
print("同一組碼的整段／分塊相同", torch.equal(whole, chunked), "樣本", len(whole))
print("形狀核對代價是有限數", bool(torch.isfinite(formal_loss)))

核對會印 `BOS／EOS／PAD 256 257 258`、時程首列 `[1,0,1]`及末列 `[4,0,1]`、分數 `(1,51,258)`與目標 `(51,)`、整段／分塊相同 `True`且4000 samples。前50步對應500 ms，第51步學EOS。輸出槽也含BOS；自由生成若誤出BOS或達上限，明確記失敗。BOS本身不納入聲音碼；正式評分與一般影音回傳仍可能解碼已產生的非空碼前綴，再依EOS與上限旗標判失敗。可播放分塊入口`sound_chunks`才在解碼前拒絕這種失敗輸出；解出部分波形不等於完成合法音型。真正自由生成只呼叫 `formal_talker.generate(program)`，不能傳入上面的人工碼列。

</details>

正式音型資料是2560 train、256 dev、200 final個完整程式家族；頻率、事件時值與停頓共同定家族，phase／振幅／雜訊只是同家族變體。兩個語義提示音明示在train中，所以A→Talker整鏈驗的是未見影音輸入接已見音型；另200個未見音型組合才驗Talker組合能力。codec固定6000次更新，Talker1000次，全部從隨機參數開始。

本輪已完成這兩個正式工作：codec有37,232個參數，dev選第5000步；Talker有109,314個參數，dev選第100步。它們仍分別跑滿6000與1000次更新並保留最佳及最後續訓狀態；選較早的checkpoint不表示後面的更新沒有執行。codec的選擇清單先固定，Talker才用這一份碼本編訓練targets，見[codec選擇紀錄](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/omni-products/v1/codec-selection.json)。能力是否達到預設門檻，另由未見家族驗收。

每碼8 bits、每秒100碼，單算聲音碼是800 bit/s，未含模型權重、音型程式或封裝。decoder每碼獨立輸出80 samples，分塊與整段逐樣本一致、未來碼擾動不改已播前綴。但本輪Talker先完成整列碼才開始decoder；首段等待包含整段碼生成，不能稱與文字交錯的即時語音。正式驗收也分別列gold音型、實際A條件的Talker、字典映射及整鏈；A錯答時不能用gold波形補上。

凍結後的真實結果分開列在下面。前兩列使用200個未見音型組合；後四列使用200個未見影音家族，映射到已在Talker訓練中出現的兩個提示音型，不能混稱同一種泛化：

| 實測工作 | 結果 |
| --- | --- |
| codec重建的有效發聲區平均SNR | 22.3752 dB，門檻10 dB |
| gold音型自由生成與格式合法 | 各200／200，音型正確門檻至少170／200 |
| 真A的圓／方語義預測 | 200／200 |
| 實際預測→音型字典 | 200／200 |
| 按實際A條件自由生成 | 200／200 |
| A→映射→Talker→decoder→分析器整鏈 | 200／200，門檻至少160／200 |

音型正確依前述分析器的頻率與時間容差判定，不要求生成波形逐樣本等於參考。例如一筆圓形提示音測得第一音結束於210 ms，目標200 ms，仍在30 ms容差內。分塊核對則是另一件事：10個預定程式的整段／分塊解碼逐樣本相同，改未來碼也沒有改動先前前綴，全部通過。

在模型已載入、CPU使用兩個Torch執行緒的10個預定程式上，首個可播放片段就緒時間中位數17.16 ms，範圍7.36–24.65 ms；其中整列碼生成的中位數已占16.93 ms，整段解碼完成中位數24.31 ms。這些時間從呼叫生成起算，未包含Python啟動、模型載入、WAV寫入或播放器。它支持這個有限生成器的內部成本說明，不能拿來對比成熟人聲模型的端到端串流延遲。完整波形、逐筆分析與原始計時見[本輪結果](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/omni-products/v1/results-summary.json)，自行產生WAV見[新版操作](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/omni-products/README.md#輸入音型程式產生有限聲音)。
